In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,IDSample,FrameNumber,J1X,J1Y,J1Z,J2X,J2Y,J2Z,J3X,J3Y,...,J23Y,J23Z,J24X,J24Y,J24Z,J25X,J25Y,J25Z,Action,Camera
0,1,1,0.164490,0.291355,3.091319,0.189296,0.615010,3.080443,0.209298,0.928650,...,0.249604,2.929706,0.237679,0.185178,2.881282,0.290780,0.236038,2.852048,3,1
1,1,2,0.164051,0.291135,3.091265,0.187257,0.614560,3.080929,0.206220,0.927991,...,0.242949,2.934857,0.237713,0.184868,2.881657,0.288819,0.236270,2.851250,3,1
2,1,3,0.160394,0.291922,3.091712,0.183053,0.614688,3.081785,0.202935,0.927368,...,0.261962,2.940350,0.237603,0.182165,2.880324,0.286495,0.221468,2.868750,3,1
3,1,4,0.157784,0.291979,3.090791,0.179416,0.614119,3.081133,0.198786,0.925982,...,0.257649,2.934111,0.231934,0.184934,2.879659,0.289556,0.229324,2.863987,3,1
4,1,5,0.154023,0.289773,3.091381,0.174946,0.611712,3.081385,0.194107,0.923457,...,0.252397,2.932880,0.223781,0.178191,2.879539,0.278711,0.216850,2.864786,3,1


In [2]:
frames = df_test.IDSample.value_counts()
ids = frames.index

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": ids,
    "answer": frames.values
})

df_1.head()

,subtaskID,datapointID,answer
0,1,4410,487
1,1,3431,478
2,1,4401,440
3,1,4206,417
4,1,4564,354


In [3]:
import numpy as np
from tqdm import tqdm

max_len = df_train.IDSample.value_counts().max()
cols = [col for col in df_train.columns if col.startswith('J')]
ids = df_train.IDSample.unique()

X = []
y1 = []
y2 = []
for idx in tqdm(ids):
    seq = []
    for _, row in df_train[df_train.IDSample == idx].iterrows():
        seq.append(row[cols].values)
    
    seq = np.array(seq)
    if len(seq) < max_len:
        pad_width = max_len - len(seq)
        seq = np.pad(seq, ((0, pad_width), (0, 0)), mode='constant')

    y1.append(df_train[df_train.IDSample == idx].Action.iloc[0])
    y2.append(df_train[df_train.IDSample == idx].Camera.iloc[0])
    X.append(seq)

X = np.array(X)
y1 = np.array(y1)
y2 = np.array(y2)

X.shape

100%|██████████████████████████████████████████████████████████████████████████████| 3354/3354 [01:43<00:00, 32.31it/s]


(3354, 516, 75)

In [4]:
ids = df_test.IDSample.unique()

X_final = []
for idx in tqdm(ids):
    seq = []
    for _, row in df_test[df_test.IDSample == idx].iterrows():
        seq.append(row[cols].values)
    
    seq = np.array(seq)
    if len(seq) < max_len:
        pad_width = max_len - len(seq)
        seq = np.pad(seq, ((0, pad_width), (0, 0)), mode='constant')

    X_final.append(seq)

X_final = np.array(X_final)

X_final.shape

100%|██████████████████████████████████████████████████████████████████████████████| 1378/1378 [00:41<00:00, 32.83it/s]


(1378, 516, 75)

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y1_train, y1_test = train_test_split(X, y1, test_size=0.2, random_state=42)

X_train.shape

(2683, 516, 75)

In [6]:
import torch

num_epochs = 100
batch_size = 32
learning_rate = 0.0005

device = torch.device("cuda")
device

device(type='cuda')

In [7]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float), torch.tensor(self.y[idx], dtype=torch.long)

train_dataset = MyDataset(X_train, y1_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MyDataset(X_test, y1_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
train_dataset[0][0][0][0]

tensor(0.2591)

In [8]:
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=75, num_layers=2, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, 5)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.fc(out)

model = LSTM().to(device)
criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model

LSTM(
  (lstm): LSTM(75, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

In [9]:
def train(epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        loss = criterion(model(inputs), labels)
        optim.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.5)
        optim.step()
        total_loss += loss.item()
    if epoch % 5 == 0:
        print("Epoch ", epoch)
        print("Training Loss ", total_loss/len(train_loader))

def test(epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            loss = criterion(model(inputs), labels)
            total_loss += loss.item()
    if epoch % 5 == 0:
        print("Validation Loss", total_loss/len(test_loader))

In [10]:
for epoch in range(1, num_epochs+1):
    train(epoch)
    test(epoch)

Epoch  5
Training Loss  0.7634218654462269
Validation Loss 0.7103960230236962
Epoch  10
Training Loss  0.6306280550502595
Validation Loss 0.604432211745353
Epoch  15
Training Loss  0.4719310328364372
Validation Loss 0.5660683569454011
Epoch  20
Training Loss  0.4600768174443926
Validation Loss 0.5645669421979359
Epoch  25
Training Loss  0.3913878007304101
Validation Loss 0.48723968650613514
Epoch  30
Training Loss  0.2849133312258692
Validation Loss 0.3506476730108261
Epoch  35
Training Loss  0.2537978772251379
Validation Loss 0.3081875095764796
Epoch  40
Training Loss  0.22358585104701065
Validation Loss 0.2519411827836718
Epoch  45
Training Loss  0.1617881816013583
Validation Loss 0.1526162682899407
Epoch  50
Training Loss  0.14739279762753063
Validation Loss 0.20636182154218355
Epoch  55
Training Loss  0.12043707946404106
Validation Loss 0.14397386004704804
Epoch  60
Training Loss  0.12519517779305933
Validation Loss 0.09887271698209502
Epoch  65
Training Loss  0.09771424309638817
V

In [11]:
class FinalDataset(Dataset):
    def __init__(self, X):
        self.X = X
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float)

final_dataset = FinalDataset(X_final)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0]

tensor(-0.7075)

In [12]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        outputs = torch.argmax(outputs, dim=1).cpu().numpy()
        preds.extend(outputs)

preds = [int(pred) for pred in preds]
preds[:10]

[2, 0, 2, 0, 0, 0, 1, 3, 1, 0]

In [13]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": ids,
    "answer": preds
})

df_2.head()

,subtaskID,datapointID,answer
0,2,3361,2
1,2,3362,0
2,2,3363,2
3,2,3364,0
4,2,3365,0


In [14]:
from torch.utils.data import Dataset, DataLoader

y2 = y2-1
X_train, X_test, y2_train, y2_test = train_test_split(X, y2, test_size=0.2, random_state=42)

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float), torch.tensor(self.y[idx], dtype=torch.long)

train_dataset = MyDataset(X_train, y2_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MyDataset(X_test, y2_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
train_dataset[0][0][0][0]

tensor(0.2591)

In [15]:
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=75, num_layers=2, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, 3)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.fc(out)

model = LSTM().to(device)
criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model

LSTM(
  (lstm): LSTM(75, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

In [16]:
for epoch in range(1, num_epochs+1):
    train(epoch)
    test(epoch)

Epoch  5
Training Loss  1.0032634075198854
Validation Loss 0.9991057571910676
Epoch  10
Training Loss  0.9529492294504529
Validation Loss 0.9449482446625119
Epoch  15
Training Loss  0.8797668339241118
Validation Loss 0.8895189137685866
Epoch  20
Training Loss  0.7918473794346764
Validation Loss 0.7685885911896115
Epoch  25
Training Loss  0.6756893087710653
Validation Loss 0.7532511098044259
Epoch  30
Training Loss  0.6736810462815421
Validation Loss 0.7678376294317699
Epoch  35
Training Loss  0.6860588962833086
Validation Loss 1.0297776999927701
Epoch  40
Training Loss  0.5825210334289641
Validation Loss 0.6756052985077813
Epoch  45
Training Loss  0.5653554733310427
Validation Loss 0.5670177226974851
Epoch  50
Training Loss  0.5502971102084432
Validation Loss 0.6490971233163562
Epoch  55
Training Loss  0.5312744691258385
Validation Loss 0.5204922329811823
Epoch  60
Training Loss  0.6222241687632742
Validation Loss 0.6500843791734605
Epoch  65
Training Loss  0.508830323637951
Validation

In [17]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        outputs = torch.argmax(outputs, dim=1).cpu().numpy()
        preds.extend(outputs)

preds = [int(pred)+1 for pred in preds]
preds[:10]

[2, 1, 1, 2, 3, 3, 3, 3, 3, 1]

In [18]:
df_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": ids,
    "answer": preds
})

df_3.head()

,subtaskID,datapointID,answer
0,3,3361,2
1,3,3362,1
2,3,3363,1
3,3,3364,2
4,3,3365,3


In [19]:
submission = pd.concat([df_1, df_2, df_3])
submission

,subtaskID,datapointID,answer
0,1,4410,487
1,1,3431,478
2,1,4401,440
3,1,4206,417
4,1,4564,354
...,...,...,...
1373,3,4734,2
1374,3,4735,2
1375,3,4736,3
1376,3,4737,3


In [20]:
submission.to_csv("submission.csv", index=False)